<a href="https://colab.research.google.com/github/Bgm777/crypto-portfolio-tracker/blob/Main/Crypto-Portfolio-Tracker.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install tabulate

In [ ]:
import requests
import json
import pandas as pd
from datetime import datetime
import os
import matplotlib.pyplot as plt
from tabulate import tabulate

class CryptoPortfolioTracker:
    def __init__(self, portfolio_file="portfolio.json"):
        self.portfolio_file = portfolio_file
        self.api_url = "https://api.coingecko.com/api/v3"
        self.portfolio = self.load_portfolio()

    def load_portfolio(self):
        """Lädt das Portfolio aus einer JSON-Datei oder erstellt ein neues."""
        if os.path.exists(self.portfolio_file):
            with open(self.portfolio_file, 'r') as file:
                return json.load(file)
        else:
            return {"coins": [], "transactions": []}

    def save_portfolio(self):
        """Speichert das Portfolio in eine JSON-Datei."""
        with open(self.portfolio_file, 'w') as file:
            json.dump(self.portfolio, file, indent=4)

    def add_transaction(self, coin_id, amount, price_per_coin, transaction_type, date=None):
        """Fügt eine neue Transaktion hinzu."""
        if date is None:
            date = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

        transaction = {
            "coin_id": coin_id,
            "amount": amount,
            "price_per_coin": price_per_coin,
            "transaction_type": transaction_type,
            "date": date,
            "value": amount * price_per_coin
        }

        self.portfolio["transactions"].append(transaction)

        # Aktualisiere oder füge Coin zum Portfolio hinzu
        coin_exists = False
        for coin in self.portfolio["coins"]:
            if coin["id"] == coin_id:
                coin_exists = True
                if transaction_type == "buy":
                    coin["amount"] += amount
                elif transaction_type == "sell":
                    coin["amount"] -= amount
                break

        if not coin_exists and transaction_type == "buy":
            coin_info = self.get_coin_info(coin_id)
            if coin_info:
                self.portfolio["coins"].append({
                    "id": coin_id,
                    "symbol": coin_info["symbol"],
                    "name": coin_info["name"],
                    "amount": amount
                })

        self.save_portfolio()
        print(f"Transaktion hinzugefügt: {transaction_type} {amount} {coin_id}")

    def get_coin_info(self, coin_id):
        """Holt Informationen zu einer Kryptowährung von der API."""
        try:
            response = requests.get(f"{self.api_url}/coins/{coin_id}")
            if response.status_code == 200:
                data = response.json()
                return {
                    "id": data["id"],
                    "symbol": data["symbol"],
                    "name": data["name"],
                    "current_price": data["market_data"]["current_price"]["usd"],
                    "price_change_24h": data["market_data"]["price_change_percentage_24h"]
                }
            else:
                print(f"Fehler: Coin {coin_id} nicht gefunden.")
                return None
        except Exception as e:
            print(f"Fehler bei der API-Anfrage: {e}")
            return None

    def get_portfolio_value(self, currency="usd"):
        """Berechnet den aktuellen Wert des gesamten Portfolios."""
        total_value = 0
        for coin in self.portfolio["coins"]:
            coin_info = self.get_coin_info(coin["id"])
            if coin_info:
                coin_value = coin["amount"] * coin_info["current_price"]
                total_value += coin_value
        return total_value

    def get_portfolio_summary(self):
        """Erstellt eine Zusammenfassung des aktuellen Portfolios."""
        summary = []
        total_value = 0
        total_cost = 0

        for coin in self.portfolio["coins"]:
            if coin["amount"] <= 0:  # Ignoriere Coins ohne Bestand
                continue

            coin_info = self.get_coin_info(coin["id"])
            if coin_info:
                current_price = coin_info["current_price"]
                current_value = coin["amount"] * current_price

                # Berechne Durchschnittskosten
                buy_transactions = [t for t in self.portfolio["transactions"]
                                   if t["coin_id"] == coin["id"] and t["transaction_type"] == "buy"]
                if buy_transactions:
                    total_cost_coin = sum(t["value"] for t in buy_transactions)
                    total_amount_bought = sum(t["amount"] for t in buy_transactions)
                    avg_buy_price = total_cost_coin / total_amount_bought if total_amount_bought > 0 else 0
                else:
                    avg_buy_price = 0

                # Profit/Loss berechnen
                cost_basis = avg_buy_price * coin["amount"]
                profit_loss = current_value - cost_basis
                profit_loss_percent = (profit_loss / cost_basis * 100) if cost_basis > 0 else 0

                total_value += current_value
                total_cost += cost_basis

                summary.append({
                    "id": coin["id"],
                    "name": coin_info["name"],
                    "symbol": coin_info["symbol"].upper(),
                    "amount": coin["amount"],
                    "avg_buy_price": avg_buy_price,
                    "current_price": current_price,
                    "value": current_value,
                    "profit_loss": profit_loss,
                    "profit_loss_percent": profit_loss_percent,
                    "price_change_24h": coin_info["price_change_24h"]
                })

        return {
            "coins": summary,
            "total_value": total_value,
            "total_cost": total_cost,
            "total_profit_loss": total_value - total_cost,
            "total_profit_loss_percent": (total_value - total_cost) / total_cost * 100 if total_cost > 0 else 0
        }

    def display_summary(self):
        """Zeigt eine Zusammenfassung des Portfolios an."""
        summary = self.get_portfolio_summary()

        if not summary["coins"]:
            print("Dein Portfolio ist leer. Füge Transaktionen hinzu, um zu beginnen.")
            return

        # Erstelle eine Tabelle für die Konsolenanzeige
        table_data = []
        for coin in summary["coins"]:
            table_data.append([
                coin["symbol"],
                coin["amount"],
                f"${coin['current_price']:.2f}",
                f"${coin['avg_buy_price']:.2f}",
                f"${coin['value']:.2f}",
                f"${coin['profit_loss']:.2f}",
                f"{coin['profit_loss_percent']:.2f}%",
                f"{coin['price_change_24h']:.2f}%"
            ])

        print("\nKRYPTO-PORTFOLIO ÜBERSICHT")
        print(tabulate(
            table_data,
            headers=["Symbol", "Menge", "Kurs", "Eink.preis", "Wert", "G/V", "G/V %", "24h %"],
            tablefmt="grid"
        ))

        print(f"\nGesamtwert: ${summary['total_value']:.2f}")
        print(f"Gesamtkosten: ${summary['total_cost']:.2f}")
        print(f"Gesamt G/V: ${summary['total_profit_loss']:.2f} ({summary['total_profit_loss_percent']:.2f}%)")

    def plot_portfolio_distribution(self):
        """Erstellt ein Kreisdiagramm der Portfolio-Verteilung."""
        summary = self.get_portfolio_summary()
        if not summary["coins"]:
            print("Kein Portfolio zum Anzeigen vorhanden.")
            return

        labels = [f"{coin['symbol']} (${coin['value']:.2f})" for coin in summary["coins"]]
        values = [coin["value"] for coin in summary["coins"]]

        plt.figure(figsize=(10, 7))
        plt.pie(values, labels=labels, autopct='%1.1f%%', startangle=140)
        plt.axis('equal')
        plt.title('Portfolio-Verteilung')
        plt.tight_layout()
        plt.savefig('portfolio_distribution.png')
        plt.show()
        print("Diagramm unter 'portfolio_distribution.png' gespeichert.")

    def plot_performance_history(self):
        """Erstellt einen historischen Performance-Verlauf basierend auf Transaktionen."""
        # Diese Funktion würde eine komplexere Implementierung erfordern,
        # um tatsächlich historische Daten zu verfolgen und zu visualisieren
        print("Diese Funktion benötigt historische Daten und ist noch nicht implementiert.")


# Beispiel für die Verwendung
if __name__ == "__main__":
    tracker = CryptoPortfolioTracker()

    # Beispiel für Benutzerinteraktion
    print("\nKRYPTO-PORTFOLIO TRACKER")
    print("======================")
    print("1. Portfolio anzeigen")
    print("2. Kauf hinzufügen")
    print("3. Verkauf hinzufügen")
    print("4. Portfolio-Verteilung anzeigen")
    print("5. Beenden")

    choice = input("\nWähle eine Option (1-5): ")

    if choice == "1":
        tracker.display_summary()
    elif choice == "2":
        coin_id = input("Coin ID (z.B. bitcoin, ethereum): ").lower()
        amount = float(input("Menge: "))
        price = float(input("Preis pro Coin in USD: "))
        tracker.add_transaction(coin_id, amount, price, "buy")
        tracker.display_summary()
    elif choice == "3":
        coin_id = input("Coin ID (z.B. bitcoin, ethereum): ").lower()
        amount = float(input("Menge: "))
        price = float(input("Preis pro Coin in USD: "))
        tracker.add_transaction(coin_id, amount, price, "sell")
        tracker.display_summary()
    elif choice == "4":
        tracker.plot_portfolio_distribution()
    elif choice == "5":
        print("Programm beendet.")
    else:
        print("Ungültige Option.")


KRYPTO-PORTFOLIO TRACKER
1. Portfolio anzeigen
2. Kauf hinzufügen
3. Verkauf hinzufügen
4. Portfolio-Verteilung anzeigen
5. Beenden


In [ ]:
# Ersetze den Hauptteil des Codes mit dieser interaktiven Schleife
if __name__ == "__main__":
    tracker = CryptoPortfolioTracker()

    while True:
        print("\nKRYPTO-PORTFOLIO TRACKER")
        print("======================")
        print("1. Portfolio anzeigen")
        print("2. Kauf hinzufügen")
        print("3. Verkauf hinzufügen")
        print("4. Portfolio-Verteilung anzeigen")
        print("5. Beenden")

        choice = input("\nWähle eine Option (1-5): ")

        if choice == "1":
            tracker.display_summary()
        elif choice == "2":
            coin_id = input("Coin ID (z.B. bitcoin, ethereum): ").lower()
            amount = float(input("Menge: "))
            price = float(input("Preis pro Coin in USD: "))
            tracker.add_transaction(coin_id, amount, price, "buy")
            tracker.display_summary()
        elif choice == "3":
            coin_id = input("Coin ID (z.B. bitcoin, ethereum): ").lower()
            amount = float(input("Menge: "))
            price = float(input("Preis pro Coin in USD: "))
            tracker.add_transaction(coin_id, amount, price, "sell")
            tracker.display_summary()
        elif choice == "4":
            tracker.plot_portfolio_distribution()
        elif choice == "5":
            print("Programm beendet.")
            break
        else:
            print("Ungültige Option.")

In [ ]:
# Ersetze den Hauptteil des Codes mit dieser interaktiven Schleife
if __name__ == "__main__":
    tracker = CryptoPortfolioTracker()

    while True:
        print("\nKRYPTO-PORTFOLIO TRACKER")
        print("======================")
        print("1. Portfolio anzeigen")
        print("2. Kauf hinzufügen")
        print("3. Verkauf hinzufügen")
        print("4. Portfolio-Verteilung anzeigen")
        print("5. Beenden")

        choice = input("\nWähle eine Option (1-5): ")

        if choice == "1":
            tracker.display_summary()
        elif choice == "2":
            coin_id = input("Coin ID (z.B. bitcoin, ethereum): ").lower()
            amount = float(input("Menge: "))
            price = float(input("Preis pro Coin in USD: "))
            tracker.add_transaction(coin_id, amount, price, "buy")
            tracker.display_summary()
        elif choice == "3":
            coin_id = input("Coin ID (z.B. bitcoin, ethereum): ").lower()
            amount = float(input("Menge: "))
            price = float(input("Preis pro Coin in USD: "))
            tracker.add_transaction(coin_id, amount, price, "sell")
            tracker.display_summary()
        elif choice == "4":
            tracker.plot_portfolio_distribution()
        elif choice == "5":
            print("Programm beendet.")
            break
        else:
            print("Ungültige Option.")